# Sprint 4 · US 4.1 — frozen wav2vec 2.0 / HuBERT / WavLM embeddings (Colab runner)

All logic lives in the repo (`src/features/ssl_embeddings.py`); this notebook only sets up Colab and calls it.
Run top to bottom. **After a disconnect:** re-run sections 1–6, then the extraction cell you were on — finished shards are skipped.

Before the first run: (1) `python experiments/scripts/pack_colab_data.py` on your machine, (2) upload `dist/esl_colab_pack.tar` to
`MyDrive/esl-pronunciation-data/`, (3) `git push` the Sprint 4 code. Runtime → Change runtime type → **T4 GPU**.

## 1 · GPU check

In [ ]:
import subprocess
out = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout
print(out or "NO GPU - Runtime > Change runtime type > T4 GPU (extraction still works on CPU but is ~20x slower)")

## 2 · Paths and Drive

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")

REPO_URL  = "https://github.com/christiannngg/esl-pronunciation-assessment.git"
PACK      = "/content/drive/MyDrive/esl-pronunciation-data/esl_colab_pack.tar"
DATA_ROOT = "/content/esl-data"                  # unpacked processed data (local Colab disk = fast)
CACHE     = "/content/ssl_cache"                 # embedding cache, ~6 GB per encoder, disposable (re-extraction takes minutes)
OUT       = "/content/drive/MyDrive/esl-pronunciation-data/frozen_outputs"   # small results only (manifests, tables, predictions)
os.makedirs(OUT, exist_ok=True)
os.environ["DATA_ROOT"] = DATA_ROOT
os.environ["SSL_CACHE_ROOT"] = CACHE

## 3 · Get the code (private repo: add a Colab secret named `GITHUB_TOKEN`)

In [ ]:
import os, subprocess
url = REPO_URL
try:
    from google.colab import userdata
    tok = userdata.get("GITHUB_TOKEN")
    if tok:
        url = REPO_URL.replace("https://", f"https://{tok}@")
except Exception:
    pass                                         # public repo or no secret: plain clone

os.chdir("/content")
if not os.path.exists("esl-pronunciation-assessment"):
    subprocess.run(["git", "clone", "--quiet", url], check=True)
os.chdir("/content/esl-pronunciation-assessment")
subprocess.run(["git", "pull", "--ff-only", "--quiet"], check=True)
print(subprocess.run(["git", "log", "--oneline", "-3"], capture_output=True, text=True).stdout)

## 4 · Dependencies (Colab already ships torch and transformers; versions are recorded in each manifest)

In [ ]:
!pip -q install soundfile pyyaml tqdm
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__, "| cuda", torch.cuda.is_available())

## 5 · Unpack the data pack onto local disk

In [ ]:
if not os.path.exists(f"{DATA_ROOT}/processed/alignments"):
    os.makedirs(DATA_ROOT, exist_ok=True)
    subprocess.run(["tar", "-xf", PACK, "-C", DATA_ROOT], check=True)
print(open(f"{DATA_ROOT}/PACK_MANIFEST.json").read())

## 6 · Dry run + unit tests (no GPU needed; validates every Sprint 3 row resolves to one pooling span)

In [ ]:
!python -m src.features.ssl_embeddings --dry-run
!pip -q install pytest
!python -m pytest tests/test_ssl_embeddings.py -q

## 7 · Smoke test — 20 utterances per dataset, one encoder (about a minute, mostly model download)

In [ ]:
!python -m src.features.ssl_embeddings --encoder wav2vec2_base --limit 20 --overwrite
!python -m src.features.ssl_embeddings --encoder wav2vec2_base --limit 20 --verify

## 8 · Full extraction — one cell per encoder (resumable)

In [ ]:
!python -m src.features.ssl_embeddings --encoder wav2vec2_base
!python -m src.features.ssl_embeddings --encoder wav2vec2_base --verify

In [ ]:
!python -m src.features.ssl_embeddings --encoder hubert_base
!python -m src.features.ssl_embeddings --encoder hubert_base --verify

In [ ]:
!python -m src.features.ssl_embeddings --encoder wavlm_base_plus
!python -m src.features.ssl_embeddings --encoder wavlm_base_plus --verify

## 9 · Keep the manifests (tiny) on Drive

In [ ]:
import glob, shutil
for m in glob.glob(f"{CACHE}/*/manifest.json"):
    name = m.split("/")[-2]
    shutil.copy(m, f"{OUT}/manifest_{name}.json")
    print("saved", name)
!du -sh {CACHE}/*